# 04b — 贝叶斯优化（下）：主动实验设计与合成配方寻优

在 `04a` 中，我们学会了用高斯过程回归（GPR）同时输出候选点的预测均值 $\mu(x)$ 和预测不确定性 $\sigma(x)$。现在我们来解决实验材料人与合成化学家最关心的核心问题：

> **假设面前有 3,955 种可能的催化反应配方（或者一片二维温度—浓度工艺窗口），每次进实验室合成与表征都要花费一整天。在绝不可能把所有配方全试一遍的前提下，如何仅用 20～25 次实验就精准锁定最高产率的配方？**

这就是**贝叶斯优化（Bayesian Optimization, BO）**与**主动学习（Active Learning）**的任务——它也是当今“自动驾驶实验室（Self-Driving Labs / Autonomous Materials Discovery）”的核心决策大脑。

本节内容安排如下：
1. **贝叶斯优化的四步闭环**：利用（Exploitation）与探索（Exploration）的权衡；
2. **手写三大采集函数（Acquisition Functions）**：纯利用（Greedy）、置信上界（UCB）与期望提升（Expected Improvement, EI）；
3. **连续工艺参数寻优实战**：直观追踪 6 轮迭代如何跳出局部最优陷阱，并拓展到二维（温度 + 浓度）参数空间；
4. **真实合成反应寻优实战**：在 Buchwald–Hartwig 钯催化碳氮偶联反应数据集（Ahneman et al., *Science* 2018，共 3,955 个反应）上，仅用 25 次实验找到 >95% 产率的最优配方；
5. **贝叶斯优化的适用边界与工程实践指南**。

---

## 0. 环境准备与数据读取辅助函数

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from sklearn.decomposition import PCA
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110

def find_file(name):
    for path in (f"data/{name}", f"../data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请确保它位于 data/ 文件夹或当前目录下")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 1. 贝叶斯优化的四步闭环：利用（Exploitation）vs 探索（Exploration）

与传统“一次性做完所有实验再拟合并分析”的被动建模不同，贝叶斯优化是一个**“建模 $\to$ 推荐下一轮实验 $\to$ 实验反馈 $\to$ 更新模型”**的序贯闭环过程：

1. **初始采样（Initialization）**：在参数空间中随机（或按拉丁超立方均匀设计）做少量几次初始实验（例如 $3 \sim 5$ 个点），构成初始观测集 $\mathcal{D}_0 = \{(x_i, y_i)\}$；
2. **拟合代理模型（Surrogate Model）**：用高斯过程回归（GPR）拟合当前已观测数据 $\mathcal{D}_t$，快速算出所有未测候选点 $x$ 的后验均值 $\mu(x)$ 和标准差 $\sigma(x)$；
3. **最大化采集函数（Acquisition Function, $\alpha(x)$）**：将 $\mu(x)$ 和 $\sigma(x)$ 代入采集函数 $\alpha(x)$，为每个候选点打出“多值得去测”的综合得分，挑出得分最高的点 $x_{t+1} = \arg\max_x \alpha(x)$ 交由实验室开展下一轮测试；
4. **更新数据集**：将新测得的实验结果 $(x_{t+1}, y_{t+1})$ 并入已观测集 $\mathcal{D}_{t+1}$，返回第 2 步循环，直到实验预算耗尽或材料性能达标。

在这个闭环中，采集函数 $\alpha(x)$ 必须精妙地平衡两种相互竞争的动机：
- **利用（Exploitation，深挖已知高地）**：去预测均值 $\mu(x)$ 很高的区域精细微调，力求稳步刷新记录；
- **探索（Exploration，开拓未知盲区）**：去预测不确定性 $\sigma(x)$ 很大的空白区探路，防止被局部小山头困住而漏掉真正的全局最高峰。

---

## 2. 手写三大经典采集函数（Greedy / UCB / EI）

假设我们的实验目标是**最大化**材料性能 $y$，目前已经做过的实验中最高性能记录为 $y_{\text{best}} = \max_{i} y_i$。最常用的三种采集函数定义如下：

1. **纯利用策略（Greedy / Pure Exploitation）**：
   $$\alpha_{\text{Greedy}}(x) = \mu(x)$$
   完全无视不确定性 $\sigma(x)$，只挑当前预测均值最高的点，极易陷入局部最优。

2. **置信上界（Upper Confidence Bound, UCB）**：
   $$\alpha_{\text{UCB}}(x) = \mu(x) + \kappa \cdot \sigma(x)$$
   直接把均值和不确定性按权重 $\kappa \ge 0$ 相加（乐观主义原则：按每个点置信区间的上界来评估其潜力）。$\kappa$ 越小越偏向利用，$\kappa$ 越大越偏向探索（常取 $\kappa = 2.0$）。

3. **期望提升（Expected Improvement, EI）**：
   衡量候选点 $x$ 的真实性能超越当前最高记录 $y_{\text{best}}$ 的**数学期望增量** $\mathbb{E}[\max(0, f(x) - y_{\text{best}} - \xi)]$。由于 $f(x) \sim \mathcal{N}(\mu(x), \sigma^2(x))$，该积分有优雅的解析闭式解：
   $$\alpha_{\text{EI}}(x) = (\mu(x) - y_{\text{best}} - \xi)\,\Phi(Z) + \sigma(x)\,\phi(Z), \quad Z = \frac{\mu(x) - y_{\text{best}} - \xi}{\sigma(x)}$$
   其中 $\Phi(\cdot)$ 和 $\phi(\cdot)$ 分别是标准正态分布的累积分布函数（`norm.cdf`）与概率密度函数（`norm.pdf`），微小常数 $\xi$（常取 `0.01`）用于调节探索倾向。注意公式右边的两项：第一项奖励高均值 $\mu(x) > y_{\text{best}}$（利用），第二项 $\sigma(x)\phi(Z)$ 奖励高不确定性（探索），无需手动拍脑袋调权重即可自动平衡两者！

下面我们构造一条具有两个峰的“未知真实工艺曲线” `hidden_process(x)`（左侧 $x=2.3$ 处有一个次优小峰，右侧 $x=7.8$ 处藏着真正的全局主峰）。假设初始 3 个实验点不幸全都落在了左侧小峰附近，我们看看 Greedy、UCB 和 EI 分别会推荐去哪里做第 4 个实验：

In [ ]:
def hidden_process(x):
    """模拟双峰材料工艺响应曲线：x=2.3 为局部次优峰 (0.6)，x=7.8 为全局主峰 (1.0)"""
    return 0.6 * np.exp(-0.5 * ((x - 2.3) / 0.8)**2) + 1.0 * np.exp(-0.5 * ((x - 7.8) / 0.9)**2)

def acq_ucb(mu, std, kappa=2.0):
    return mu + kappa * std

def acq_ei(mu, std, y_best, xi=0.01):
    with np.errstate(divide="ignore", invalid="ignore"):
        imp = mu - y_best - xi
        Z = np.where(std > 1e-9, imp / std, 0.0)
        ei = np.where(std > 1e-9, imp * norm.cdf(Z) + std * norm.pdf(Z), 0.0)
    return ei

X_grid = np.linspace(0, 10, 300).reshape(-1, 1)
X_init = np.array([[1.2], [2.2], [3.6]])   # 初始 3 个实验点全部集中在左侧小峰附近
y_init = hidden_process(X_init).ravel()

gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-5, random_state=0)
gp.fit(X_init, y_init)
mu, std = gp.predict(X_grid, return_std=True)

fig, axes = plt.subplots(2, 1, figsize=(9.5, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1.2]})
axes[0].plot(X_grid, hidden_process(X_grid), "k--", alpha=0.5, label="Hidden True Process")
axes[0].plot(X_grid, mu, "b-", lw=2, label="GP Posterior Mean")
axes[0].fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.15, label="95% CI")
axes[0].scatter(X_init, y_init, c="red", s=60, zorder=5, label="Initial Experiments (n = 3)")
axes[0].set_ylabel("Device Performance")
axes[0].set_title("GP Surrogate Model vs Three Acquisition Functions")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

ei_vals  = acq_ei(mu, std, y_init.max(), xi=0.01)
ucb_vals = acq_ucb(mu, std, kappa=2.0)

axes[1].plot(X_grid, ei_vals / (ei_vals.max() + 1e-9), "g-", lw=2, label=f"EI (recommends x = {X_grid[np.argmax(ei_vals), 0]:.2f})")
axes[1].plot(X_grid, ucb_vals / (ucb_vals.max() + 1e-9), "m-", lw=1.8, label=f"UCB (recommends x = {X_grid[np.argmax(ucb_vals), 0]:.2f})")
axes[1].plot(X_grid, mu / (mu.max() + 1e-9), "b:", lw=1.5, label=f"Greedy (recommends x = {X_grid[np.argmax(mu), 0]:.2f})")
axes[1].set_xlabel("Process Parameter x (e.g., Normalized Annealing Temperature)")
axes[1].set_ylabel("Normalized Acquisition")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3. 连续材料工艺寻优：直观追踪 6 轮迭代跳出局部最优

从上图下半部分可以清楚看到：
- **Greedy（蓝色点线）**被左侧已知小山峰牢牢吸住，推荐下一个点继续测 $x = 2.34$，永远不会去右侧看一眼；
- **EI（绿线）和 UCB（紫线）**则注意到右侧 $x > 4.5$ 的广阔空白区具有很大的不确定性 $\sigma(x)$，果断给出高分推荐去右侧探路。

下面我们把 **EI 贝叶斯优化** 连续运行 6 轮（`Step 1` 至 `Step 6`），看看每测完一个新点（绿色五角星），GPR 的置信带如何收缩、并一步步锁定右侧 $x = 7.8$ 的全局最高峰：

In [ ]:
X_obs = X_init.copy()
y_obs = y_init.copy()

fig, axes = plt.subplots(2, 3, figsize=(13, 6.5), sharex=True, sharey=True)
for step, ax in enumerate(axes.ravel(), start=1):
    gp_step = GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-5, random_state=0)
    gp_step.fit(X_obs, y_obs)
    mu, std = gp_step.predict(X_grid, return_std=True)
    ei = acq_ei(mu, std, y_obs.max(), xi=0.01)
    next_x = X_grid[np.argmax(ei)]
    next_y = hidden_process(next_x).item()

    ax.plot(X_grid, hidden_process(X_grid), "k--", alpha=0.4)
    ax.plot(X_grid, mu, "b-", lw=1.5)
    ax.fill_between(X_grid.ravel(), mu - 1.96 * std, mu + 1.96 * std, color="blue", alpha=0.15)
    ax.scatter(X_obs, y_obs, c="red", s=35, zorder=5)
    ax.scatter([next_x[0]], [next_y], c="lime", edgecolors="black", s=85, marker="*", zorder=6, label=f"Next x = {next_x[0]:.2f}")
    ax.set_title(f"Step {step} | Best y = {max(y_obs.max(), next_y):.3f}")
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.25)

    X_obs = np.vstack([X_obs, next_x.reshape(1, 1)])
    y_obs = np.append(y_obs, next_y)

plt.suptitle("Bayesian Optimization (EI) Escaping Local Optimum in 6 Steps", fontsize=12)
plt.tight_layout()
plt.show()

可以看到：第 1–2 步 EI 先在左侧山峰确认顶点并试探右侧边界，第 3 步跨入右侧未知区测到 $x = 6.22$ 发现数值回升，第 4–6 步便迅速顺藤摸瓜登顶 $x = 7.80$ 的全局最优（`Best y = 1.000`）！

### 🏋️ 练习 04b-1：调节 UCB 探索权重 $\kappa$ 对跳出局部最优的影响

把上面 6 步循环中的采集函数从 `acq_ei` 换成 `acq_ucb(mu, std, kappa=...)`，分别测试三个不同的探索系数：
- `kappa = 0.1`（极度保守，几乎只做局部利用）；
- `kappa = 2.0`（利用与探索平衡）；
- `kappa = 10.0`（高度鼓励探索未知空白区）。

打印每个 `kappa` 在 6 步迭代后找到的最大值 `y_obs.max()` 以及这 6 步选中的采样位置，观察为什么 `kappa = 0.1` 会被死死困在左侧小山峰。

In [ ]:
# 在下方填写你的代码：


### 🏋️ 练习 04b-2：二维材料工艺窗口寻优（退火温度 $T$ + 前驱体浓度 $c$）

将一维工艺推广到二维工艺平面：下方函数 `process_2d(X)` 模拟了由温度 $T \in [0, 10]$ 和浓度 $c \in [0, 10]$ 共同决定的薄膜性能（在 $(2.5, 3.0)$ 处有一个高为 `0.65` 的次优峰，在 $(7.5, 7.2)$ 处有一个高为 `1.00` 的全局主峰），并在 $25 \times 25 = 625$ 个候选网格点 `X_cand` 上计算好了真实性能 `y_true_all`。
1. 从随机抽取的 3 个初始点 `init_idx` 出发，用 `GaussianProcessRegressor(kernel=C(1.0) * RBF(1.5), alpha=1e-5)` 配合 `acq_ei` 循环迭代 **15 步**（总计 18 次实验），每次从未测网格点 `unseen` 中挑选 EI 最大的点加入观测集 `obs_idx`；
2. 对比 18 次贝叶斯优化找到的最大性能与随机盲抽 18 个点找到的最大性能，并在二维等高线图上标出初始点（白圈）和 BO 选中的 15 个点（红星）。

In [ ]:
def process_2d(X):
    t, c = X[:, 0], X[:, 1]
    peak1 = 0.65 * np.exp(-0.3 * ((t - 2.5)**2 + (c - 3.0)**2))
    peak2 = 1.00 * np.exp(-0.35 * ((t - 7.5)**2 + (c - 7.2)**2))
    return peak1 + peak2

g = np.linspace(0, 10, 25)
TT, CC = np.meshgrid(g, g)
X_cand = np.column_stack([TT.ravel(), CC.ravel()])
y_true_all = process_2d(X_cand)

# 在下方填写你的 15 步二维 BO 循环与绘图代码：


---

## 4. 真实化学合成寻优实战：Buchwald–Hartwig 钯催化偶联反应

在合成化学与配方工程中，实验条件往往不是连续数值，而是**多个离散组分构成的组合空间（Combinatorial Space）**：例如从候选库中挑选 1 种膦配体（Ligand）、1 种碱（Base）、1 种添加剂（Additive）和 1 种芳基卤代物底物（Aryl Halide），组合成一个具体的反应配方。

`data/bh-reactions.csv` 收录了 Doyle 与 Dreher 团队发表在 *Science* 2018 上的经典高通量实验数据集（Ahneman et al., *Science* **360**, 186–190, 2018），包含 **3,955 个真实的钯催化碳氮偶联反应**及其测得的实验产率 `yield`（`0% ~ 100%`）。
- 在全部 3,955 个反应中，平均产率只有 **33.1%**，但最优配方的产率高达 **99.9%**。
- 如果我们在一个新课题中只能做 **25 次实验**（仅占 3,955 种可能组合的 **0.6%**），把每个反应的 4 个组分做 **One-Hot 独热编码**（共 42 维 0/1 向量，表示用了哪号配体、哪号碱、哪号添加剂、哪号底物），用 GPR + EI/UCB 驱动贝叶斯优化，能不能在 25 次实验内揪出产率 >95% 的顶级配方？

下面我们用 5 个不同的随机初始种子（每次随机测 5 个初始反应 + 20 轮 BO 主动推荐 = 共 25 次实验），对比 **随机盲试（Random Search）**、**BO (EI)** 与 **BO (UCB)** 的平均寻优曲线：

In [ ]:
bh = pd.read_csv(find_file("bh-reactions.csv"))
print(f"反应总数: {len(bh)} | 全库平均产率: {bh['yield'].mean():.1f}% | 全库最高产率: {bh['yield'].max():.1f}%")

# 对 4 个离散反应组分做 One-Hot 编码（共 42 维特征）
X_oh = pd.get_dummies(bh[["ligand", "additive", "base", "aryl halide"]], dtype=float).values
y_yield = bh["yield"].values
print("One-Hot 反应组合特征矩阵形状:", X_oh.shape)

def run_bo_reaction(X, y, n_init=5, n_steps=20, seed=0, strategy="ei"):
    rng = np.random.RandomState(seed)
    n_total = len(y)
    observed = list(rng.choice(n_total, size=n_init, replace=False))
    best_history = [y[observed].max()]

    for _ in range(n_steps):
        if strategy == "random":
            unseen = [i for i in range(n_total) if i not in observed]
            next_idx = rng.choice(unseen)
        else:
            kernel = C(1.0) * RBF(length_scale=2.0) + WhiteKernel(noise_level=1.0)
            gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, random_state=seed)
            gp.fit(X[observed], y[observed])

            unseen_idx = np.array([i for i in range(n_total) if i not in observed])
            mu, std = gp.predict(X[unseen_idx], return_std=True)
            if strategy == "ei":
                scores = acq_ei(mu, std, y[observed].max(), xi=0.5)
            elif strategy == "ucb":
                scores = acq_ucb(mu, std, kappa=2.0)
            next_idx = unseen_idx[np.argmax(scores)]

        observed.append(next_idx)
        best_history.append(y[observed].max())
    return np.array(best_history)

seeds = [0, 1, 2, 3, 4]
curves_rand = np.array([run_bo_reaction(X_oh, y_yield, seed=s, strategy="random") for s in seeds])
curves_ei   = np.array([run_bo_reaction(X_oh, y_yield, seed=s, strategy="ei") for s in seeds])
curves_ucb  = np.array([run_bo_reaction(X_oh, y_yield, seed=s, strategy="ucb") for s in seeds])

steps = np.arange(5, 26)
plt.figure(figsize=(8.5, 4.8))
for name, mat, col in [
    ("Random Search", curves_rand, "gray"),
    ("BO (Expected Improvement)", curves_ei, "green"),
    ("BO (UCB, kappa=2)", curves_ucb, "blue"),
]:
    mean_c = mat.mean(axis=0)
    std_c  = mat.std(axis=0)
    plt.plot(steps, mean_c, lw=2.2, color=col, label=f"{name} (25-exp mean best = {mean_c[-1]:.1f}%)")
    plt.fill_between(steps, mean_c - std_c, mean_c + std_c, color=col, alpha=0.12)

plt.axhline(y_yield.max(), color="red", ls="--", lw=1, label=f"Global Max ({y_yield.max():.1f}%)")
plt.xlabel("Total Number of Experiments Conducted")
plt.ylabel("Best Reaction Yield Found So Far (%)")
plt.title("Buchwald-Hartwig Coupling Optimization (3,955 Candidate Reactions)")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

在 3,955 个候选反应中，仅开展 25 次实验（不到全空间的 `0.65%`），贝叶斯优化（EI 与 UCB）就能把找到的最高产率稳定推进到 **94% ～ 97%**，而相同预算下的随机盲试还停留在 80% 左右。

### 🏋️ 练习 04b-3：用化学结构指纹（Morgan FP + PCA）替代纯编号 One-Hot 编码

上面使用的 One-Hot 编码只告诉模型“这是 1 号配体、那是 2 号配体”，模型完全不知道两个配体在化学结构上是否相似。
- `bh` 表格中的 `ligand`、`additive`、`base`、`aryl halide` 四列本身存放的就是各组分的 **SMILES 字符串**！
- 请为这 4 列分别计算 256 维的 Morgan 二值指纹并横向拼接（共 1,024 维），用 `PCA(n_components=30, random_state=0)` 降维到 30 维得到特征矩阵 `X_fp_pca`；
- 在 `seeds = [0, 1, 2]` 上运行 `run_bo_reaction(X_fp_pca, y_yield, strategy="ei")`，对比它与 One-Hot 编码在 25 次实验后的平均最高产率。

In [ ]:
# 在下方填写你的代码：


---

## 5. 贝叶斯优化的适用边界与工程实践指南

在将贝叶斯优化用于你自己的实验课题之前，请牢记以下三条工程边界：

1. **适用维度边界（低维至中维最优）**：
   标准高斯过程回归基于欧氏距离或核相似度，最适合 **1 ～ 20 维** 的连续工艺参数（温度、退火时间、旋涂转速、前驱体配比、退火气氛压强）或经 PCA 压缩后的反应描述符。如果直接输入上千维未压缩特征，高维空间中所有点对距离趋同，GPR 的后验方差会退化为常数，此时应先用第三讲的 PCA 降维，或改用基于树模型的代理优化器（如 SMAC / 随机森林 BO）。
2. **样本规模边界（小样本至中等样本）**：
   标准 GPR 每次拟合需要对 $N \times N$ 核矩阵求逆，时间复杂度为 $\mathcal{O}(N^3)$。当累计实验点数 $N < 1000$ 时，每次迭代仅需几毫秒到几秒；当数据量超过数千条时，标准 GPR 会明显变慢，此时通常已走出“小样本盲区”，可切换为深度神经网络集成或稀疏高斯过程。
3. **并行孔板/多通道实验（Batch BO）**：
   上面的标准循环每次只推荐 1 个最优实验点（Sequential BO）。但如果你的实验室一次可以平行合成 8 个样品或跑一整块 24 孔板，直接取 EI 排名前 8 的点往往会导致这 8 个点全挤在同一个峰值附近。工程上常采用 **克里金假想观测策略（Kriging Believer）**：选出第 1 个点后，把它的预测均值 $\mu(x_1)$ 暂时当作“假想真实观测值”塞回 GPR 更新方差（迫使 $x_1$ 周围的 $\sigma$ 归零），再选第 2 个点，依此类推即可一次性生成兼顾高性能与空间分散性的批次推荐清单。

---

## 小结：第四讲（贝叶斯优化）知识全景

| 模块 | 核心工具与概念 | 解决的实际科研问题 |
|---|---|---|
| **`04a` 高斯过程回归** | 多元正态条件分布、`GaussianProcessRegressor`、`RBF`/`Matern`/`WhiteKernel`、`Tanimoto` 分子核 | 在小样本实验数据上同时输出预测均值 $\mu(x)$ 与不确定性 $\sigma(x)$，自动剥离测量噪声并提供原生适用域预警 |
| **`04b` 贝叶斯优化** | 四步主动学习闭环、利用 vs 探索权衡、`Greedy` / `UCB` / `EI` 采集函数 | 在昂贵的材料工艺窗口与数千种催化反应组合中，仅用数十次实验跳出局部最优并锁定全局最优配方 |

### 后续课程衔接

至此，我们已经完成了课程上半学期前四讲（**Python 与 RDKit 数据基础 → 监督学习 → 无监督学习 → 贝叶斯优化与主动实验设计**）的完整闭环。在接下来的模块中，我们将进一步深入特征工程与 SHAP 可解释性分析（`05`）、无机晶体材料信息学（`06`），以及下半学期的深度学习、SMILES 语言模型、分子生成模型（VAE）与图神经网络（GNN & 机器学习势函数）。